# ProboSed 01 — JCORES VCD Mining and MTD Catalog
**IODP Expedition 405 — Site C0019J, Japan Trench**

Extracts stability scores from the JCORES VCD PDF and produces the MTD boundary catalog.

### CORC0019.pdf page format:
```
Hole C0019J Core 7K, interval 139 to 142.805 m (core depth below seafloor)
```
Core ID extracted from `Core 7K`, depth from `interval 139 to ...`

### Output files:
- `C0019J_VCD_stability_log.csv`
- `C0019J_MTD_catalog.csv`
- `C0019J_stability_profile.png`

In [1]:
# Cell 1 — Install dependencies
!pip install pymupdf openpyxl pandas numpy matplotlib --quiet

In [2]:
# Cell 2 — Mount Google Drive and clone ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

repo_path = '/content/ProboSed'
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

result = subprocess.run(
    ['git', 'clone', 'https://github.com/rocknrene/ProboSed.git', repo_path],
    capture_output=True, text=True
)
print('Clone return code:', result.returncode)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Add to path — do this here so all subsequent cells can import
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print('ProboSed contents:', os.listdir(repo_path))
print('core_ml exists:', os.path.exists(os.path.join(repo_path, 'core_ml')))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clone return code: 0
ProboSed contents: ['requirements.txt', 'Pyproject.toml', '__init__.py', 'LICENSE', 'notebooks', 'core_ml', 'geochem', 'README.md', '.gitignore', '.git', 'slope', 'utils', 'CITATION.cff', 'transport']
core_ml exists: True


In [3]:
# Cell 3 — Verify import and check lexicon
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml.labeler import JCORESMiner, JCORES_LEXICON, DRILLING_CAUSE_TERMS

print('Lexicon checks:')
print(f'  minor in lexicon:          {"minor" in JCORES_LEXICON}  (should be False)')
print(f'  disturb in lexicon:        {"disturb" in JCORES_LEXICON}  (should be False)')
print(f'  sheared in lexicon:        {"sheared" in JCORES_LEXICON}  (should be True)')
print(f'  high disturbance:          {"high disturbance" in JCORES_LEXICON}  (should be True)')
print(f'  due to drilling in causes: {"due to drilling" in DRILLING_CAUSE_TERMS}  (should be True)')
print(f'\nCORE_PATTERN:  {JCORESMiner.CORE_PATTERN.pattern}')
print(f'DEPTH_PATTERN: {JCORESMiner.DEPTH_PATTERN.pattern}')

Lexicon checks:
  minor in lexicon:          False  (should be False)
  disturb in lexicon:        True  (should be False)
  sheared in lexicon:        True  (should be True)
  high disturbance:          True  (should be True)
  due to drilling in causes: True  (should be True)

CORE_PATTERN:  (?:Core\s+(\d+[A-Z])|(?:405-)?C\d{4}[A-Z]-(\d+[A-Z]))
DEPTH_PATTERN: interval\s+(\d+\.?\d*)\s+to\s+\d+\.?\d*\s*m


In [4]:
# Cell 4 — Configure paths
import os

DATA_DIR   = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

VCD_PDF      = os.path.join(DATA_DIR, 'vcds', 'CORC0019.pdf')
SUMMARY_XLSX = os.path.join(DATA_DIR, '405-C0019J_CoreSummary_2410280924.xlsx')
STABILITY_CSV = os.path.join(OUTPUT_DIR, 'C0019J_VCD_stability_log.csv')
MTD_CSV       = os.path.join(OUTPUT_DIR, 'C0019J_MTD_catalog.csv')
PROFILE_PNG   = os.path.join(OUTPUT_DIR, 'C0019J_stability_profile.png')

for path, label in [(VCD_PDF, 'VCD PDF'), (SUMMARY_XLSX, 'Core summary Excel')]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — check path'
    print(f'{label}: {status}')
    print(f'  {path}')

VCD PDF: FOUND
  /content/drive/MyDrive/iodp/X405/Data & Data Tracking/vcds/CORC0019.pdf
Core summary Excel: FOUND
  /content/drive/MyDrive/iodp/X405/Data & Data Tracking/405-C0019J_CoreSummary_2410280924.xlsx


In [5]:
# Cell 5 — Verify PDF text format
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

import fitz
from core_ml.labeler import JCORESMiner

doc = fitz.open(VCD_PDF)
print(f'PDF pages: {len(doc)}')
print()

for i in range(min(3, len(doc))):
    page = doc.load_page(i)
    text = page.get_text()
    lines = [l.strip() for l in text.split('\n') if l.strip()][:4]
    print(f'Page {i+1} header:')
    for l in lines:
        print(f'  {l}')
    core_m  = JCORESMiner.CORE_PATTERN.search(text)
    depth_m = JCORESMiner.DEPTH_PATTERN.search(text)
    core_id = (core_m.group(1) or core_m.group(2)).upper() if core_m else None
    depth   = float(depth_m.group(1)) if depth_m else None
    print(f'  -> Core={core_id}  Depth={depth}')
    print()

doc.close()

if core_id is None:
    print('WARNING: Core ID not found. Check VCD_PDF path.')
else:
    print('Core ID and depth extraction working correctly.')

PDF pages: 125

Page 1 header:
  HS
  PMAG
  SS
  SS
  -> Core=1K  Depth=82.0

Page 2 header:
  PP
  HS
  CARB,XRD,XRF
  SS
  -> Core=2K  Depth=91.5

Page 3 header:
  PP
  CARB,XRD,XRF
  SS
  SS
  -> Core=3K  Depth=101.0

Core ID and depth extraction working correctly.


In [7]:
# Cell 6 — Build depth backbone
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml.labeler import JCORESMiner

miner    = JCORESMiner(VCD_PDF, SUMMARY_XLSX)
backbone = miner.build_backbone()

print(f'Backbone sample (first 10 rows):')
print(backbone.head(10).to_string(index=False))

Backbone built: 491 section slots, deepest = 829.09 m CSF-A
Backbone sample (first 10 rows):
Core Section  Depth_m    Type
  1K       1   82.000 Section
  1K       2   83.500 Section
  1K       3   85.000 Section
  1K       4   85.255 Section
  1K      CC   85.255      CC
  2K       1   91.500 Section
  2K       2   93.000 Section
  2K       3   94.500 Section
  2K       4   95.430 Section
  2K      CC   95.430      CC


In [13]:
# Cell 7 — Fix lexicon, extract stability scores, tag by hole, save outputs
#
# This cell is fully self-contained and can be rerun after a session reset.
# It performs four operations in sequence:
#
#   1. Fixes the JCORES_LEXICON in memory based on spot-check validation:
#        REMOVED: terms that fired as false positives on every page
#        ADDED:   terms found during manual spot-check of ten cores
#
#   2. Builds the depth backbone and runs the miner on CORC0019.pdf.
#      The PDF contains five holes: C0019J (pp 1-88), C0019K (pp 89-105),
#      C0019L (p 106), C0019M (pp 107-120), C0019P (pp 121-125).
#
#   3. Extracts the hole identifier from every page header and adds a
#      'Hole' column so all holes are preserved and filterable.
#      Page header format: 'Hole C0019J Core 7K, interval 139 to 142.805 m'
#
#   4. Saves two CSV files:
#        C0019_VCD_stability_log_allholes.csv — all holes, tagged by Hole column
#        C0019J_VCD_stability_log.csv         — C0019J only, for downstream notebooks
#
# Note: lexicon fixes are applied in memory only. To make them permanent,
# update JCORES_LEXICON in labeler.py on GitHub and push.

import sys, os, re
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

import fitz
from core_ml import labeler as lm
from core_ml.labeler import JCORESMiner

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR      = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR    = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
VCD_PDF       = os.path.join(DATA_DIR, 'vcds', 'CORC0019.pdf')
SUMMARY_XLSX  = os.path.join(DATA_DIR, '405-C0019J_CoreSummary_2410280924.xlsx')
ALL_HOLES_CSV = os.path.join(OUTPUT_DIR, 'C0019_VCD_stability_log_allholes.csv')
STABILITY_CSV = os.path.join(OUTPUT_DIR, 'C0019J_VCD_stability_log.csv')

for path, label in [(VCD_PDF, 'VCD PDF'), (SUMMARY_XLSX, 'Core summary')]:
    print(f'{label}: {"FOUND" if os.path.exists(path) else "NOT FOUND"}')
print()

# ── Step 1: Fix lexicon in memory ─────────────────────────────────────────────
# These changes correct false positives identified during spot-check validation
# against ten cores distributed across the full depth range of C0019J.
#
# REMOVED terms (fired on every page or on section header text):
#   'disturb'            — substring of 'drilling disturbance', fired everywhere
#   'disturbed'          — same problem
#   'color banding'      — matched 'Color banded clay' in lithology headers
#   'colour banding'     — same, British spelling variant
#   'soft deformation'   — appeared in figure captions and adjacent descriptions
#   'soft sediment deform' — same
#
# ADDED terms (found in actual disturbance descriptions during spot-check):
#   'shear deformation'      — core 87K: 'extensive evidence of shear deformation'
#   'evidence of shear'      — same page
#   'soft-sediment deform'   — hyphenated variant found in some core descriptions
#   'some evidence of deform' — core 50K section-level text

for term in ['disturb', 'disturbed', 'color banding',
             'colour banding', 'soft deformation',
             'soft sediment deform']:
    if term in lm.JCORES_LEXICON:
        del lm.JCORES_LEXICON[term]
        print(f'Removed: "{term}"')

lm.JCORES_LEXICON['shear deformation']       = 1
lm.JCORES_LEXICON['evidence of shear']       = 1
lm.JCORES_LEXICON['soft-sediment deform']    = 1
lm.JCORES_LEXICON['some evidence of deform'] = 1
lm.JCORES_LEXICON['high distubance'] = 0   # typo in core 60K VCD text
print('Added: shear deformation variants')
print()
print('Active score-0 terms:', [k for k, v in lm.JCORES_LEXICON.items() if v == 0])
print('Active score-1 terms:', [k for k, v in lm.JCORES_LEXICON.items() if v == 1])
print()

# ── Step 2: Build backbone and extract ────────────────────────────────────────
# The backbone provides authoritative CSF-A depths from the core summary Excel.
# The miner extracts core ID and depth directly from the page header text and
# falls back to the backbone if the header depth is not found.
miner    = JCORESMiner(VCD_PDF, SUMMARY_XLSX)
backbone = miner.build_backbone()
vcd_df   = miner.extract(backbone)

# ── Step 3: Add Hole column ───────────────────────────────────────────────────
# CORC0019.pdf contains five holes from Site C0019.
# The hole identifier is extracted from each page header:
#   'Hole C0019J Core 7K, interval 139 to 142.805 m (core depth below seafloor)'
# All holes are preserved in the output — filter by Hole column as needed.
HOLE_PATTERN = re.compile(r'Hole\s+(C\d+[A-Z]+)', re.IGNORECASE)
doc = fitz.open(VCD_PDF)
hole_map = {}
for i in range(len(doc)):
    text = doc.load_page(i).get_text()
    match = HOLE_PATTERN.search(text)
    if match:
        hole_map[i + 1] = match.group(1)   # PDF_Page is 1-indexed
doc.close()

vcd_df.insert(2, 'Hole', vcd_df['PDF_Page'].map(hole_map))

print(f'Total rows extracted: {len(vcd_df)}')
print(f'\nRows per hole:')
print(vcd_df.groupby('Hole')['PDF_Page'].count().to_string())
print()

# ── Step 4: Save outputs ──────────────────────────────────────────────────────
# All holes — for future cross-hole analysis (C0019K deep zone, C0019M shallow)
vcd_df.to_csv(ALL_HOLES_CSV, index=False)
print(f'Saved all holes: {ALL_HOLES_CSV}')

# C0019J only — for Chapter I MTD catalog and spot-check notebook
j_df = vcd_df[vcd_df['Hole'] == 'C0019J'].reset_index(drop=True)
print(f'C0019J rows: {len(j_df)}')  # should be 88
j_df.to_csv(STABILITY_CSV, index=False)
print(f'Saved C0019J only: {STABILITY_CSV}')
j_df = vcd_df[vcd_df['Hole'] == 'C0019J'].reset_index(drop=True)
print(f'C0019J rows: {len(j_df)}')  # should be 88
print()

print(f'\nC0019J score distribution:')
print(j_df['Stability'].value_counts().sort_index())
print(f'\nDepth-matched: {j_df["Depth_m"].notna().sum()} / {len(j_df)}')
print(f'Drilling artifacts: {j_df["Drilling_Artifact"].sum()}')

print(f'\nSpot-check cores (C0019J):')
for core in ['1K', '14K', '40K', '50K', '60K', '75K', '87K']:
    rows = j_df[j_df['Core'] == core]
    if not rows.empty:
        r = rows.iloc[0]
        print(f'  {core:4s} ({r["Depth_m"]:6.1f} mbsf): '
              f'score={r["Stability"]}  '
              f'term="{r["Disturbance"]}"  '
              f'drilling={r["Drilling_Artifact"]}')
    else:
        print(f'  {core}: not found')

VCD PDF: FOUND
Core summary: FOUND

Added: shear deformation variants

Active score-0 terms: ['soupy', 'slurried', 'flow-in', 'highly disturbed', 'heavily disturbed', 'completely disturbed', 'high disturbance', 'high disturb', 'void', 'gouge', 'clay gouge', 'chaotic', 'high distubance']
Active score-1 terms: ['moderately disturbed', 'biscuit', 'fall-in', 'fractured', 'brecciated', 'sheared', 'scaly', 'deformed fabric', 'deformed', 'shear deformation', 'evidence of shear', 'soft-sediment deform', 'some evidence of deform']

Backbone built: 491 section slots, deepest = 829.09 m CSF-A
Extracting from 125 pages...
Extraction complete: 125 pages, 125 depth-matched (100%)
Total rows extracted: 125

Rows per hole:
Hole
C0019J    88
C0019K    17
C0019L     1
C0019M    14
C0019P     5

Saved all holes: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019_VCD_stability_log_allholes.csv
C0019J rows: 88
Saved C0019J only: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_VCD_stability_log.

In [8]:
# Cell 8 — Save stability log
vcd_df.to_csv(STABILITY_CSV, index=False)
print(f'Saved: {STABILITY_CSV}')
print(f'Rows: {len(vcd_df)}')

Saved: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_VCD_stability_log.csv
Rows: 125


In [9]:
# Cell 9 — Build MTD catalog
# Drilling artifact intervals excluded before catalog construction.
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml.labeler import JCORESMiner

geo_df      = vcd_df[~vcd_df['Drilling_Artifact']].copy()
mtd_catalog = JCORESMiner.score_to_mtd_catalog(geo_df, stability_threshold=1)

mtd_catalog.to_csv(MTD_CSV, index=False)
print(f'MTD intervals identified: {len(mtd_catalog)}')
print(f'Saved: {MTD_CSV}')
print()
if len(mtd_catalog) > 0:
    print(mtd_catalog.to_string(index=False))

MTD intervals identified: 18
Saved: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_MTD_catalog.csv

mtd_id  top_m  bottom_m  thickness_m  mean_stability
 MTD-1    6.5      16.0          9.5             0.0
 MTD-2   74.0      82.0          8.0             0.0
 MTD-3  105.0     110.5          5.5             0.0
 MTD-4  205.5     215.0          9.5             0.0
 MTD-5  253.0     262.5          9.5             1.0
 MTD-6  272.0     291.0         19.0             0.0
 MTD-7  300.5     310.0          9.5             1.0
 MTD-8  406.0     415.5          9.5             0.0
 MTD-9  425.0     453.5         28.5             1.0
MTD-10  473.0     549.0         76.0             0.0
MTD-11  558.5     577.5         19.0             0.5
MTD-12  606.5     616.0          9.5             1.0
MTD-13  644.5     654.0          9.5             0.0
MTD-14  702.0     721.0         19.0             0.0
MTD-15  825.0     828.0          3.0             1.0
MTD-16  833.0     839.0          6.0       

In [10]:
# Cell 10 — Plot stability profile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plot_df = vcd_df[~vcd_df['Drilling_Artifact']].dropna(
    subset=['Depth_m','Stability']
).sort_values('Depth_m')

fig, ax = plt.subplots(figsize=(6, 16))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f7f7f7')

ax.step(plot_df['Stability'], plot_df['Depth_m'],
        where='post', color='#2c3e50', lw=1.5, zorder=3)

for _, row in mtd_catalog.iterrows():
    ax.axhspan(row['top_m'], row['bottom_m'],
               alpha=0.2, color='#cc2222', zorder=1)

ax.axhline(820, color='#ff4444', ls='--', lw=1.5, alpha=0.8,
           label='Plate boundary fault (~820 mbsf)')

ax.invert_yaxis()
ax.set_xlim(-0.5, 3.5)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(
    ['Slurried\n(0)', 'Scaly\n(1)', 'Coherent\n(2)', 'Intact\n(3)'],
    fontsize=10
)
ax.set_ylabel('Depth (m CSF-A)', fontsize=12)
ax.set_title('C0019J Stability Profile\n(JCORESMiner extraction)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.text(0.02, 0.02, f'MTD intervals: {len(mtd_catalog)}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#cc2222', alpha=0.2))

plt.tight_layout()
plt.savefig(PROFILE_PNG, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {PROFILE_PNG}')

Saved: /content/drive/MyDrive/iodp/X405/ProboSed_Output/C0019J_stability_profile.png


In [11]:
# Cell 11 — Summary
print('=== MTD CATALOG SUMMARY ===')
print(f'Total MTD intervals:          {len(mtd_catalog)}')
if len(mtd_catalog) > 0:
    print(f'Shallowest top:               {mtd_catalog["top_m"].min():.1f} m CSF-A')
    print(f'Deepest base:                 {mtd_catalog["bottom_m"].max():.1f} m CSF-A')
    print(f'Mean thickness:               {mtd_catalog["thickness_m"].mean():.1f} m')
    print(f'Total MTD thickness:          {mtd_catalog["thickness_m"].sum():.1f} m')
    print()
    print('Individual intervals:')
    for _, row in mtd_catalog.iterrows():
        print(f'  {row["mtd_id"]}: {row["top_m"]:.1f} - '
              f'{row["bottom_m"]:.1f} m  '
              f'({row["thickness_m"]:.1f} m thick, '
              f'mean stability = {row["mean_stability"]:.1f})')
print()
print(f'Drilling artifacts excluded:   {vcd_df["Drilling_Artifact"].sum()}')
print(f'Depth-unmatched pages:         {vcd_df["Depth_m"].isna().sum()}')
print()
print('Output files:')
print(f'  {STABILITY_CSV}')
print(f'  {MTD_CSV}')
print(f'  {PROFILE_PNG}')
print()
print('Next: open notebook 07 (spot-check), reload Cell 3, rerun Cells 4 and 5.')

=== MTD CATALOG SUMMARY ===
Total MTD intervals:          18
Shallowest top:               6.5 m CSF-A
Deepest base:                 944.0 m CSF-A
Mean thickness:               15.1 m
Total MTD thickness:          271.5 m

Individual intervals:
  MTD-1: 6.5 - 16.0 m  (9.5 m thick, mean stability = 0.0)
  MTD-2: 74.0 - 82.0 m  (8.0 m thick, mean stability = 0.0)
  MTD-3: 105.0 - 110.5 m  (5.5 m thick, mean stability = 0.0)
  MTD-4: 205.5 - 215.0 m  (9.5 m thick, mean stability = 0.0)
  MTD-5: 253.0 - 262.5 m  (9.5 m thick, mean stability = 1.0)
  MTD-6: 272.0 - 291.0 m  (19.0 m thick, mean stability = 0.0)
  MTD-7: 300.5 - 310.0 m  (9.5 m thick, mean stability = 1.0)
  MTD-8: 406.0 - 415.5 m  (9.5 m thick, mean stability = 0.0)
  MTD-9: 425.0 - 453.5 m  (28.5 m thick, mean stability = 1.0)
  MTD-10: 473.0 - 549.0 m  (76.0 m thick, mean stability = 0.0)
  MTD-11: 558.5 - 577.5 m  (19.0 m thick, mean stability = 0.5)
  MTD-12: 606.5 - 616.0 m  (9.5 m thick, mean stability = 1.0)
  MTD-13: